# pandas : tabular data for AI and data science
    1. Series            ---> 1D labeled array , index alignment
    2. DataFrame         ---> creation , shape , dtypes , info , describe
    3. selection         ---> [] , loc , iloc , boolean masks , query , Copy-on-Write
    4. new columns       ---> vectorized math , np.select , cut / qcut , map , assign
    5. missing data      ---> isna , dropna , fillna , group-wise imputation , interpolate
    6. dtypes & cleaning ---> to_numeric , to_datetime , category , nullable Int64 , Persian digits
    7. text              ---> the .str accessor , regex , rule-based routing
    8. sort & count      ---> sort_values , nlargest , value_counts , duplicates
    9. groupby           ---> split-apply-combine , agg , transform , pivot_table , crosstab
   10. combine tables    ---> merge , validate , indicator , concat
   11. reshape           ---> melt , pivot , stack / unstack
   12. time series       ---> .dt , resample , rolling , shift , pct_change
   13. files             ---> csv , json lines , parquet , chunks
   14. performance       ---> iterrows vs apply vs vectorized , memory
   15. applications      ---> RFM segmentation , forecasting features , target-encoding leakage ,
                              sensor anomalies , rule-table expert system , model evaluation by slice

In [1]:
import numpy as np
import pandas as pd
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)
print("pandas", pd.__version__, "| numpy", np.__version__)

pandas 3.0.5 | numpy 2.5.3


## 1. Series : a 1D NumPy array + an index (labels) + a name

In [2]:
s = pd.Series([12.5, 18.0, 15.2, 20.1], index=["Tehran", "Isfahan", "Shiraz", "Tabriz"], name="temp")
print(s)
print("dtype:", s.dtype, "| index:", list(s.index))
print("by label s['Shiraz'] =", s["Shiraz"], "| by position s.iloc[0] =", s.iloc[0])
print(s[s > 15])                    # boolean mask , like NumPy
print((s * 9 / 5 + 32).round(1))    # vectorized , like NumPy
print(type(s.to_numpy()), s.to_numpy())

Tehran     12.5
Isfahan    18.0
Shiraz     15.2
Tabriz     20.1
Name: temp, dtype: float64
dtype: float64 | index: ['Tehran', 'Isfahan', 'Shiraz', 'Tabriz']
by label s['Shiraz'] = 15.2 | by position s.iloc[0] = 12.5
Isfahan    18.0
Shiraz     15.2
Tabriz     20.1
Name: temp, dtype: float64
Tehran     54.5
Isfahan    64.4
Shiraz     59.4
Tabriz     68.2
Name: temp, dtype: float64
<class 'numpy.ndarray'> [12.5 18.  15.2 20.1]


In [3]:
# index ALIGNMENT : pandas matches LABELS , NumPy matches POSITIONS
sales_jan = pd.Series({"apple": 100, "banana": 80, "cherry": 50})
sales_feb = pd.Series({"banana": 90, "cherry": 60, "date": 30})

print(sales_jan + sales_feb)                      # missing label on one side -> NaN
print(sales_jan.add(sales_feb, fill_value=0))     # treat missing as 0
print("positional (WRONG here):", sales_jan.to_numpy() + sales_feb.to_numpy())

apple       NaN
banana    170.0
cherry    110.0
date        NaN
dtype: float64
apple     100.0
banana    170.0
cherry    110.0
date       30.0
dtype: float64
positional (WRONG here): [190 140  80]


## 2. DataFrame : a table = a dict of Series that share one index
    each COLUMN has ONE dtype , different columns can have different dtypes

In [4]:
df = pd.DataFrame({
    "name":   ["Ali", "Sara", "Reza", "Mina", "Omid", "Neda"],
    "city":   ["Tehran", "Shiraz", "Tehran", "Isfahan", "Shiraz", "Tehran"],
    "age":    [34, 28, 45, 31, 39, 26],
    "salary": [52.0, 61.5, 75.0, 48.0, np.nan, 58.5],     # million toman per month
    "dept":   ["AI", "AI", "Data", "Data", "AI", "Web"],
})
print(df)
print("shape:", df.shape)
print(df.dtypes)

   name     city  age  salary  dept
0   Ali   Tehran   34    52.0    AI
1  Sara   Shiraz   28    61.5    AI
2  Reza   Tehran   45    75.0  Data
3  Mina  Isfahan   31    48.0  Data
4  Omid   Shiraz   39     NaN    AI
5  Neda   Tehran   26    58.5   Web
shape: (6, 5)
name          str
city          str
age         int64
salary    float64
dept          str
dtype: object


In [5]:
df.info()
print(df.describe())                      # numeric columns only
print(df.describe(include="all").T[["count", "unique", "top", "mean"]])

<class 'pandas.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   name    6 non-null      str    
 1   city    6 non-null      str    
 2   age     6 non-null      int64  
 3   salary  5 non-null      float64
 4   dept    6 non-null      str    
dtypes: float64(1), int64(1), str(3)
memory usage: 372.0 bytes
             age     salary
count   6.000000   5.000000
mean   33.833333  59.000000
std     7.139094  10.398317
min    26.000000  48.000000
25%    28.750000  52.000000
50%    32.500000  58.500000
75%    37.750000  61.500000
max    45.000000  75.000000
       count unique     top       mean
name       6      6     Ali        NaN
city       6      3  Tehran        NaN
age      6.0    NaN     NaN  33.833333
salary   5.0    NaN     NaN       59.0
dept       6      3      AI        NaN


In [6]:
# other ways to build a DataFrame
records = [                                               # list of dicts : JSON from an API
    {"id": 1, "label": "cat", "score": 0.91},
    {"id": 2, "label": "dog", "score": 0.78},
    {"id": 3, "label": "cat"},                            # missing key -> NaN
]
print(pd.DataFrame(records))

arr = np.arange(12).reshape(4, 3)                         # NumPy array + column names
print(pd.DataFrame(arr, columns=["f1", "f2", "f3"]))
print("head(2):\n", df.head(2), "\ntail(1):\n", df.tail(1))

   id label  score
0   1   cat   0.91
1   2   dog   0.78
2   3   cat    NaN
   f1  f2  f3
0   0   1   2
1   3   4   5
2   6   7   8
3   9  10  11
head(2):
    name    city  age  salary dept
0   Ali  Tehran   34    52.0   AI
1  Sara  Shiraz   28    61.5   AI 
tail(1):
    name    city  age  salary dept
5  Neda  Tehran   26    58.5  Web


## 3. Selection
    df["col"]            ---> one column  (Series)
    df[["c1", "c2"]]     ---> several columns (DataFrame)
    df.loc[rows, cols]   ---> by LABEL     (slice end INCLUDED)
    df.iloc[rows, cols]  ---> by POSITION  (slice end EXCLUDED , like Python)

In [7]:
print(type(df["age"]).__name__, "|", type(df[["age"]]).__name__)
print(df[["name", "age"]])
print("loc[2, 'name'] =", df.loc[2, "name"], "| iloc[2, 0] =", df.iloc[2, 0])
print("loc[1:3] -> rows", df.loc[1:3].index.tolist(), "(end included)")
print("iloc[1:3] -> rows", df.iloc[1:3].index.tolist(), "(end excluded)")

Series | DataFrame
   name  age
0   Ali   34
1  Sara   28
2  Reza   45
3  Mina   31
4  Omid   39
5  Neda   26
loc[2, 'name'] = Reza | iloc[2, 0] = Reza
loc[1:3] -> rows [1, 2, 3] (end included)
iloc[1:3] -> rows [1, 2] (end excluded)


In [8]:
people = df.set_index("name")                 # a meaningful index
print(people.loc["Reza"])                     # one row -> Series
print(people.loc[["Ali", "Neda"], ["city", "salary"]])
print(people.loc["Reza":"Omid", "age"])       # label slice : Reza , Mina , Omid

city      Tehran
age           45
salary      75.0
dept        Data
Name: Reza, dtype: object
        city  salary
name                
Ali   Tehran    52.0
Neda  Tehran    58.5
name
Reza    45
Mina    31
Omid    39
Name: age, dtype: int64


In [9]:
# boolean filtering : use & | ~ and PARENTHESES (not and / or)
mask = (df["age"] > 30) & (df["city"] == "Tehran")
print(df.loc[mask, ["name", "age", "city"]])

print(df[df["dept"].isin(["AI", "Web"])]["name"].tolist())
print(df[df["age"].between(28, 35)]["name"].tolist())             # both ends included
print(df.query("age > 30 and dept == 'AI'")[["name", "age"]])

min_age = 35
print(df.query("age >= @min_age")["name"].tolist())               # @ = a Python variable

try:
    df[df["age"] > 30 and df["city"] == "Tehran"]
except ValueError as e:
    print("ValueError:", str(e)[:60], "...")

   name  age    city
0   Ali   34  Tehran
2  Reza   45  Tehran
['Ali', 'Sara', 'Omid', 'Neda']
['Ali', 'Sara', 'Mina']
   name  age
0   Ali   34
4  Omid   39
['Reza', 'Omid']
ValueError: The truth value of a Series is ambiguous. Use a.empty, a.boo ...


In [10]:
# Copy-on-Write : a filtered result behaves like a NEW object
import warnings
work = df.copy()

tehran = work[work["city"] == "Tehran"]
tehran["salary"] = 0                                   # changes only `tehran`
print("original unchanged:", work["salary"].tolist())

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    work["salary"][0] = 999                            # chained assignment : never works
print("chained assignment ->", work.loc[0, "salary"], "|", [w.category.__name__ for w in caught])

work.loc[work["city"] == "Tehran", "salary"] = 0      # the correct way : ONE .loc call
print("with .loc:", work["salary"].tolist())

original unchanged: [52.0, 61.5, 75.0, 48.0, nan, 58.5]
chained assignment -> 52.0 | ['ChainedAssignmentError']
with .loc: [0.0, 61.5, 0.0, 48.0, nan, 0.0]


## 4. Creating columns : always think VECTORIZED (like NumPy)

In [11]:
df["salary_year"] = df["salary"] * 12
df["senior"] = df["age"] >= 35
df["level"] = np.select(
    [df["age"] < 30, df["age"] < 40],
    ["junior", "mid"],
    default="senior",
)
df["age_group"] = pd.cut(df["age"], bins=[0, 30, 40, 100], labels=["<30", "30-39", "40+"], right=False)
df["city_code"] = df["city"].map({"Tehran": "THR", "Shiraz": "SYZ", "Isfahan": "IFN"})
print(df)

   name     city  age  salary  dept  salary_year  senior   level age_group city_code
0   Ali   Tehran   34    52.0    AI        624.0   False     mid     30-39       THR
1  Sara   Shiraz   28    61.5    AI        738.0   False  junior       <30       SYZ
2  Reza   Tehran   45    75.0  Data        900.0    True  senior       40+       THR
3  Mina  Isfahan   31    48.0  Data        576.0   False     mid     30-39       IFN
4  Omid   Shiraz   39     NaN    AI          NaN    True     mid     30-39       SYZ
5  Neda   Tehran   26    58.5   Web        702.0   False  junior       <30       THR


In [12]:
# cut = fixed edges , qcut = equal-COUNT groups (quantiles)
scores = pd.Series(np.random.default_rng(0).normal(70, 12, 1000))
print(pd.cut(scores, bins=4).value_counts().sort_index())
print(pd.qcut(scores, q=4, labels=["Q1", "Q2", "Q3", "Q4"]).value_counts().sort_index())

(23.123, 44.103]      18
(44.103, 65.0]       334
(65.0, 85.896]       567
(85.896, 106.792]     81
Name: count, dtype: int64
Q1    250
Q2    250
Q3    250
Q4    250
Name: count, dtype: int64


In [13]:
# assign : build new columns in a chain without changing df
summary = (
    df.drop(columns=["salary_year", "senior", "level", "age_group", "city_code"])
      .assign(salary_per_age=lambda d: (d["salary"] / d["age"]).round(2),
              name_upper=lambda d: d["name"].str.upper())
      .sort_values("salary_per_age", ascending=False)
)
print(summary)
df = df[["name", "city", "age", "salary", "dept"]]          # back to the original columns

   name     city  age  salary  dept  salary_per_age name_upper
5  Neda   Tehran   26    58.5   Web            2.25       NEDA
1  Sara   Shiraz   28    61.5    AI            2.20       SARA
2  Reza   Tehran   45    75.0  Data            1.67       REZA
3  Mina  Isfahan   31    48.0  Data            1.55       MINA
0   Ali   Tehran   34    52.0    AI            1.53        ALI
4  Omid   Shiraz   39     NaN    AI             NaN       OMID


## A realistic dataset : customers of an online subscription service
    plan , monthly spend , support calls , tenure ... and whether they CHURNED (left)

In [14]:
rng = np.random.default_rng(42)
n = 2000
plan = rng.choice(["basic", "pro", "premium"], n, p=[0.6, 0.3, 0.1])
tenure = rng.integers(1, 61, n)                                    # months
calls = rng.poisson(np.where(plan == "basic", 2.0, 1.0))
base_price = pd.Series(plan).map({"basic": 200, "pro": 450, "premium": 900}).to_numpy()   # thousand toman
logit = -1.0 + 0.45 * calls - 0.05 * tenure + 0.5 * (plan == "basic")

customers = pd.DataFrame({
    "customer_id": np.arange(1001, 1001 + n),
    "city": rng.choice(["Tehran", "Isfahan", "Shiraz", "Tabriz", "Mashhad"], n, p=[0.4, 0.15, 0.15, 0.15, 0.15]),
    "age": rng.integers(18, 70, n).astype(float),
    "plan": plan,
    "tenure_months": tenure,
    "monthly_spend": (base_price * rng.lognormal(0, 0.25, n)).round(0),
    "support_calls": calls,
    "churned": rng.random(n) < 1 / (1 + np.exp(-logit)),
})
truth = customers.copy()                                            # kept to grade imputation later
customers.loc[rng.choice(n, 120, replace=False), "age"] = np.nan
customers.loc[rng.choice(n, 80, replace=False), "monthly_spend"] = np.nan
print(customers.head())
print(customers.shape, "| churn rate:", round(customers["churned"].mean(), 3))

   customer_id     city   age   plan  tenure_months  monthly_spend  support_calls  churned
0         1001   Tehran  27.0    pro             20          425.0              2     True
1         1002  Mashhad  25.0  basic             51          176.0              1    False
2         1003   Tehran  46.0    pro              3          548.0              2    False
3         1004  Mashhad  50.0    pro             27          500.0              0    False
4         1005   Tehran  24.0  basic             49          168.0              4     True
(2000, 8) | churn rate: 0.21


## 5. Missing data
    1. MEASURE it   2. understand WHY it is missing   3. drop , fill , or flag

In [15]:
print(customers.isna().sum())
print("share missing:\n", customers.isna().mean().round(3)[lambda s: s > 0])
print("rows with any NaN:", customers.isna().any(axis=1).sum())
print("dropna rows left :", len(customers.dropna()), "of", len(customers))

customer_id        0
city               0
age              120
plan               0
tenure_months      0
monthly_spend     80
support_calls      0
churned            0
dtype: int64
share missing:
 age              0.06
monthly_spend    0.04
dtype: float64
rows with any NaN: 195
dropna rows left : 1805 of 2000


In [16]:
# imputation : global median vs median WITHIN each plan  (spend depends strongly on plan)
miss = customers["monthly_spend"].isna()

global_fill = customers["monthly_spend"].fillna(customers["monthly_spend"].median())
group_fill = customers["monthly_spend"].fillna(
    customers.groupby("plan")["monthly_spend"].transform("median")
)

true_vals = truth.loc[miss, "monthly_spend"]
print("MAE global median:", round((global_fill[miss] - true_vals).abs().mean(), 1))
print("MAE plan median  :", round((group_fill[miss] - true_vals).abs().mean(), 1))

MAE global median: 168.6
MAE plan median  : 77.6


In [17]:
# keep the information that a value WAS missing , then fill
clean = customers.assign(
    age_missing=customers["age"].isna(),
    age=customers["age"].fillna(customers["age"].median()),
    monthly_spend=group_fill,
)
print(clean.isna().sum().sum(), "NaN left")
print(clean[clean["age_missing"]].head(3))

# time-ordered data : interpolate between neighbours
sensor = pd.Series([20.0, 20.5, np.nan, np.nan, 22.0, 22.4])
print("interpolate:", sensor.interpolate().tolist())
print("ffill      :", sensor.ffill().tolist())

0 NaN left
    customer_id     city   age   plan  tenure_months  monthly_spend  support_calls  churned  age_missing
66         1067   Tehran  44.0  basic             52          250.0              1    False         True
77         1078   Tehran  44.0    pro             37          696.0              1    False         True
83         1084  Mashhad  44.0  basic             48          160.0              0    False         True
interpolate: [20.0, 20.5, 21.0, 21.5, 22.0, 22.4]
ffill      : [20.0, 20.5, 20.5, 20.5, 22.0, 22.4]


## 6. dtypes and cleaning messy columns

In [18]:
raw = pd.DataFrame({
    "price": ["1,250,000", "980000", "۲۵۰۰۰۰۰", "N/A", " 75000 "],
    "qty":   ["3", "5", "", "2", "1"],
    "date":  ["2026-01-05", "2026/02/10", "2026-03-15", "not a date", "2026-05-01"],
})
print(raw.dtypes)

persian_digits = str.maketrans("۰۱۲۳۴۵۶۷۸۹٠١٢٣٤٥٦٧٨٩", "01234567890123456789")
fixed = raw.assign(
    price=pd.to_numeric(raw["price"].str.translate(persian_digits).str.replace(",", "").str.strip(),
                        errors="coerce"),
    qty=pd.to_numeric(raw["qty"], errors="coerce").astype("Int64"),
    date=pd.to_datetime(raw["date"], format="mixed", errors="coerce"),
)
print(fixed)
print(fixed.dtypes)

price    str
qty      str
date     str
dtype: object
       price   qty       date
0  1250000.0     3 2026-01-05
1   980000.0     5 2026-02-10
2  2500000.0  <NA> 2026-03-15
3        NaN     2        NaT
4    75000.0     1 2026-05-01
price           float64
qty               Int64
date     datetime64[us]
dtype: object


In [19]:
# int + NaN : float64 by default , or the nullable Int64
print(pd.Series([1, None, 3]).dtype, "|", pd.Series([1, None, 3], dtype="Int64").tolist())

# category : few distinct values repeated many times
city = customers["city"]
cat = city.astype("category")
print("memory str     :", city.memory_usage(deep=True) // 1024, "KB")
print("memory category:", cat.memory_usage(deep=True) // 1024, "KB")
print("categories:", list(cat.cat.categories), "| codes:", cat.cat.codes.head(5).tolist())

customers["plan"] = pd.Categorical(customers["plan"], categories=["basic", "pro", "premium"], ordered=True)
print("ordered:", (customers["plan"] > "basic").sum(), "customers above basic")

float64 | [1, <NA>, 3]
memory str     : 108 KB
memory category: 2 KB
categories: ['Isfahan', 'Mashhad', 'Shiraz', 'Tabriz', 'Tehran'] | codes: [4, 1, 4, 1, 4]
ordered: 805 customers above basic


## 7. Text : the `.str` accessor (vectorized string methods)

In [20]:
tickets = pd.Series([
    "  Login FAILED after update ",
    "payment error: card declined",
    "App crashes on login screen",
    "Refund request #4521",
    "PAYMENT page not loading",
    "how do I change my plan?",
])
t = tickets.str.strip().str.lower()
print(t.tolist())
print("contains login|payment:", t.str.contains("login|payment").tolist())
print("ticket numbers:", t.str.extract(r"#(\d+)")[0].tolist())
print("word counts   :", t.str.split().str.len().tolist())

['login failed after update', 'payment error: card declined', 'app crashes on login screen', 'refund request #4521', 'payment page not loading', 'how do i change my plan?']
contains login|payment: [True, True, True, False, True, False]
ticket numbers: [nan, nan, nan, '4521', nan, nan]
word counts   : [4, 4, 5, 3, 4, 6]


In [21]:
# a tiny rule-based router (expert system) : first matching rule wins
route = np.select(
    [t.str.contains(r"refund|payment|card"),
     t.str.contains(r"login|password"),
     t.str.contains(r"crash|not loading|error")],
    ["billing", "account", "technical"],
    default="general",
)
print(pd.DataFrame({"ticket": t, "team": route}))

                         ticket     team
0     login failed after update  account
1  payment error: card declined  billing
2   app crashes on login screen  account
3          refund request #4521  billing
4      payment page not loading  billing
5      how do i change my plan?  general


## 8. Sorting , counting , duplicates

In [22]:
print(customers.sort_values(["plan", "monthly_spend"], ascending=[True, False])
               .groupby("plan", observed=True).head(1)[["customer_id", "plan", "monthly_spend"]])
print(customers.nlargest(3, "monthly_spend")[["customer_id", "plan", "monthly_spend"]])
print(customers["city"].value_counts())
print(customers["city"].value_counts(normalize=True).round(3))
print("distinct cities:", customers["city"].nunique())

      customer_id     plan  monthly_spend
227          1228    basic          427.0
754          1755      pro         1270.0
1873         2874  premium         1720.0
      customer_id     plan  monthly_spend
1873         2874  premium         1720.0
1947         2948  premium         1674.0
1618         2619  premium         1478.0
city
Tehran     795
Shiraz     331
Tabriz     297
Mashhad    293
Isfahan    284
Name: count, dtype: int64
city
Tehran     0.398
Shiraz     0.166
Tabriz     0.148
Mashhad    0.146
Isfahan    0.142
Name: proportion, dtype: float64
distinct cities: 5


In [23]:
log = pd.DataFrame({
    "user": ["u1", "u2", "u1", "u3", "u2", "u1"],
    "event": ["login", "login", "login", "buy", "login", "buy"],
    "ts": pd.to_datetime(["2026-11-01 10:00", "2026-11-01 10:01", "2026-11-01 10:00",
                          "2026-11-01 10:05", "2026-11-01 10:09", "2026-11-01 10:12"]),
})
print("exact duplicates:", log.duplicated().sum())
print(log.drop_duplicates())
print("last event per user:\n", log.sort_values("ts").drop_duplicates(subset="user", keep="last"))

exact duplicates: 1
  user  event                  ts
0   u1  login 2026-11-01 10:00:00
1   u2  login 2026-11-01 10:01:00
3   u3    buy 2026-11-01 10:05:00
4   u2  login 2026-11-01 10:09:00
5   u1    buy 2026-11-01 10:12:00
last event per user:
   user  event                  ts
3   u3    buy 2026-11-01 10:05:00
4   u2  login 2026-11-01 10:09:00
5   u1    buy 2026-11-01 10:12:00


## 9. groupby : split -> apply -> combine
    agg       ---> one row per group
    transform ---> same shape as the input (group statistic broadcast back to each row)

In [24]:
print(customers.groupby("plan", observed=True)["monthly_spend"].mean().round(1))

report = customers.groupby("plan", observed=True).agg(
    customers=("customer_id", "size"),
    avg_spend=("monthly_spend", "mean"),
    avg_calls=("support_calls", "mean"),
    churn_rate=("churned", "mean"),
).round(3)
print(report)

plan
basic      204.4
pro        469.4
premium    908.0
Name: monthly_spend, dtype: float64
         customers  avg_spend  avg_calls  churn_rate
plan                                                
basic         1195    204.423      1.904       0.268
pro            596    469.417      1.002       0.122
premium        209    908.005      1.053       0.129


In [25]:
# churn rate by number of support calls : a strong signal
by_calls = customers.groupby(customers["support_calls"].clip(upper=6))["churned"].agg(["size", "mean"]).round(3)
by_calls.index = by_calls.index.map(lambda c: "6+" if c == 6 else str(c))
print(by_calls)

               size   mean
support_calls             
0               452  0.093
1               680  0.181
2               456  0.252
3               239  0.251
4               112  0.393
5                36  0.639
6+               25  0.520


In [26]:
# transform : compare each customer with their own plan
customers["spend_vs_plan"] = (
    customers["monthly_spend"] / customers.groupby("plan", observed=True)["monthly_spend"].transform("mean")
).round(2)
print(customers[["customer_id", "plan", "monthly_spend", "spend_vs_plan"]].head())

# filter : keep only cities with high churn
print(customers.groupby("city")["churned"].mean().round(3).sort_values(ascending=False))
high = customers.groupby("city").filter(lambda g: g["churned"].mean() > 0.22)
print("cities kept by filter (churn > 22%):", sorted(high["city"].unique()), "|", len(high), "rows")

   customer_id   plan  monthly_spend  spend_vs_plan
0         1001    pro          425.0           0.91
1         1002  basic          176.0           0.86
2         1003    pro          548.0           1.17
3         1004    pro          500.0           1.07
4         1005  basic          168.0           0.82
city
Isfahan    0.243
Mashhad    0.225
Tehran     0.214
Tabriz     0.189
Shiraz     0.178
Name: churned, dtype: float64
cities kept by filter (churn > 22%): ['Isfahan', 'Mashhad'] | 577 rows


In [27]:
print(customers.pivot_table(index="city", columns="plan", values="churned", aggfunc="mean", observed=True).round(2))
print(pd.crosstab(customers["plan"], customers["churned"], normalize="index").round(3))
print(pd.crosstab(customers["plan"], customers["churned"], margins=True))

plan     basic   pro  premium
city                         
Isfahan   0.29  0.18     0.18
Mashhad   0.29  0.08     0.27
Shiraz    0.21  0.14     0.09
Tabriz    0.22  0.15     0.12
Tehran    0.30  0.10     0.07
churned  False  True 
plan                 
basic    0.732  0.268
pro      0.878  0.122
premium  0.871  0.129
churned  False  True   All
plan                      
basic      875   320  1195
pro        523    73   596
premium    182    27   209
All       1580   420  2000


## 10. Combining tables : merge (SQL JOIN) and concat

In [28]:
users = pd.DataFrame({"user_id": [1, 2, 3, 4], "name": ["Ali", "Sara", "Reza", "Mina"]})
orders = pd.DataFrame({"user_id": [2, 3, 3, 5], "amount": [120, 75, 40, 300]})

for how in ["inner", "left", "outer"]:
    m = users.merge(orders, on="user_id", how=how, indicator=True)
    print(f"--- {how} ({len(m)} rows)\n", m)

--- inner (3 rows)
    user_id  name  amount _merge
0        2  Sara     120   both
1        3  Reza      75   both
2        3  Reza      40   both
--- left (5 rows)
    user_id  name  amount     _merge
0        1   Ali     NaN  left_only
1        2  Sara   120.0       both
2        3  Reza    75.0       both
3        3  Reza    40.0       both
4        4  Mina     NaN  left_only
--- outer (6 rows)
    user_id  name  amount      _merge
0        1   Ali     NaN   left_only
1        2  Sara   120.0        both
2        3  Reza    75.0        both
3        3  Reza    40.0        both
4        4  Mina     NaN   left_only
5        5   NaN   300.0  right_only


In [29]:
# validate : catch wrong assumptions about keys
try:
    users.merge(orders, on="user_id", validate="one_to_one")
except pd.errors.MergeError as e:
    print("MergeError:", e)
print(len(users.merge(orders, on="user_id", validate="one_to_many")), "rows : one user -> many orders OK")

# many-to-many explosion : duplicated keys on BOTH sides multiply rows
a = pd.DataFrame({"k": [1, 1, 1], "x": [1, 2, 3]})
b = pd.DataFrame({"k": [1, 1, 1, 1], "y": [1, 2, 3, 4]})
print("3 rows x 4 rows ->", len(a.merge(b, on="k")), "rows")

MergeError: Merge keys are not unique in right dataset; not a one-to-one merge
Duplicates in right:
  user_id
       3 ...
3 rows : one user -> many orders OK
3 rows x 4 rows -> 12 rows


In [30]:
jan = pd.DataFrame({"product": ["A", "B"], "sales": [10, 20]})
feb = pd.DataFrame({"product": ["A", "C"], "sales": [15, 5]})
both = pd.concat([jan.assign(month="Jan"), feb.assign(month="Feb")], ignore_index=True)
print(both)
print(pd.concat([jan, feb], axis=1, keys=["jan", "feb"]))

  product  sales month
0       A     10   Jan
1       B     20   Jan
2       A     15   Feb
3       C      5   Feb
      jan           feb      
  product sales product sales
0       A    10       A    15
1       B    20       C     5


## 11. Reshaping : wide <-> long
    long (tidy) : one row per observation  -> best for groupby , plotting , ML
    wide        : one row per entity        -> best for reading , some models

In [31]:
wide = pd.DataFrame({"store": ["S1", "S2"], "Jan": [100, 80], "Feb": [120, 90], "Mar": [130, 70]})
long = wide.melt(id_vars="store", var_name="month", value_name="sales")
print(long)
back = long.pivot(index="store", columns="month", values="sales")[["Jan", "Feb", "Mar"]]
print(back)

stacked = back.stack()
print(stacked.head(4))
print(stacked.unstack().equals(back))

  store month  sales
0    S1   Jan    100
1    S2   Jan     80
2    S1   Feb    120
3    S2   Feb     90
4    S1   Mar    130
5    S2   Mar     70
month  Jan  Feb  Mar
store               
S1     100  120  130
S2      80   90   70
store  month
S1     Jan      100
       Feb      120
       Mar      130
S2     Jan       80
dtype: int64
True


## 12. Time series

In [32]:
rng = np.random.default_rng(7)
n_tx = 8000
start = pd.Timestamp("2026-01-01")
offsets = pd.to_timedelta(rng.integers(0, 304 * 24 * 60, n_tx), unit="min")    # Jan 1 -> Oct 31
tx = pd.DataFrame({
    "ts": start + offsets,
    "customer_id": rng.choice(customers["customer_id"], n_tx),
    "amount": rng.lognormal(5.5, 0.6, n_tx).round(0),                         # thousand toman
}).sort_values("ts", ignore_index=True)

tx["month"] = tx["ts"].dt.month
tx["weekday"] = tx["ts"].dt.day_name()
tx["hour"] = tx["ts"].dt.hour
tx["iran_weekend"] = tx["ts"].dt.dayofweek.isin([3, 4])                       # Thursday , Friday
print(tx.head())
print(tx["ts"].min(), "->", tx["ts"].max())

                   ts  customer_id  amount  month   weekday  hour  iran_weekend
0 2026-01-01 01:19:00         1835   549.0      1  Thursday     1          True
1 2026-01-01 01:53:00         1702   353.0      1  Thursday     1          True
2 2026-01-01 02:01:00         2431   601.0      1  Thursday     2          True
3 2026-01-01 02:44:00         1252   142.0      1  Thursday     2          True
4 2026-01-01 04:19:00         1049   171.0      1  Thursday     4          True
2026-01-01 01:19:00 -> 2026-10-31 22:20:00


In [33]:
daily = tx.set_index("ts").resample("D")["amount"].sum()
print("days:", len(daily))
monthly = tx.set_index("ts").resample("MS")["amount"].agg(["count", "sum"])
print(monthly)

feat = pd.DataFrame({"sales": daily})
feat["lag_1"] = feat["sales"].shift(1)                      # yesterday
feat["ma_7"] = feat["sales"].shift(1).rolling(7).mean()     # past 7 days , no look-ahead
feat["pct_change"] = feat["sales"].pct_change().round(3)
print(feat.iloc[6:11].round(2))

days: 304
            count       sum
ts                         
2026-01-01    766  219714.0
2026-02-01    732  219169.0
2026-03-01    868  246971.0
2026-04-01    824  247103.0
2026-05-01    797  224423.0
2026-06-01    757  232145.0
2026-07-01    798  239901.0
2026-08-01    835  249566.0
2026-09-01    834  244504.0
2026-10-01    789  239550.0
             sales   lag_1     ma_7  pct_change
ts                                             
2026-01-07  4934.0  7095.0      NaN       -0.30
2026-01-08  4714.0  4934.0  6954.57       -0.04
2026-01-09  4598.0  4714.0  6292.86       -0.02
2026-01-10  9385.0  4598.0  5869.00        1.04
2026-01-11  7187.0  9385.0  6392.57       -0.23


## 13. Reading and writing files

In [34]:
from pathlib import Path
out = Path("pandas_demo")
out.mkdir(exist_ok=True)

tx.head(1000).to_csv(out / "tx.csv", index=False)
back = pd.read_csv(out / "tx.csv")
print("after csv  :", back["ts"].dtype, "<- dates became text !")
back = pd.read_csv(out / "tx.csv", parse_dates=["ts"], dtype={"customer_id": "int32"})
print("with parse :", back["ts"].dtype, back["customer_id"].dtype)

customers.head(3).to_json(out / "customers.jsonl", orient="records", lines=True)
print((out / "customers.jsonl").read_text().splitlines()[0])
print(pd.read_json(out / "customers.jsonl", lines=True).shape)

try:
    tx.to_parquet(out / "tx.parquet")
    print("parquet ok :", pd.read_parquet(out / "tx.parquet")["ts"].dtype)
except ImportError:
    print("parquet needs pyarrow : conda install pyarrow")

after csv  : str <- dates became text !
with parse : datetime64[us] int32
{"customer_id":1001,"city":"Tehran","age":27.0,"plan":"pro","tenure_months":20,"monthly_spend":425.0,"support_calls":2,"churned":true,"spend_vs_plan":0.91}
(3, 9)
parquet needs pyarrow : conda install pyarrow


In [35]:
# a file too big for memory : process it in chunks
tx.to_csv(out / "tx_full.csv", index=False)
total = 0.0
rows = 0
for chunk in pd.read_csv(out / "tx_full.csv", chunksize=2000):
    total += chunk["amount"].sum()
    rows += len(chunk)
print(rows, "rows | total:", total, "| check:", total == tx["amount"].sum())

CLEANUP = True
if CLEANUP:
    for f in out.iterdir():
        f.unlink()
    out.rmdir()

8000 rows | total: 2363046.0 | check: True


## 14. Performance : iterrows < apply < vectorized

In [36]:
import time
big = pd.DataFrame({"price": rng.uniform(10, 100, 100_000), "qty": rng.integers(1, 10, 100_000)})

t0 = time.perf_counter()
r1 = [row["price"] * row["qty"] for _, row in big.iterrows()]
t1 = time.perf_counter()
r2 = big.apply(lambda row: row["price"] * row["qty"], axis=1)
t2 = time.perf_counter()
r3 = big["price"] * big["qty"]
t3 = time.perf_counter()
print(f"iterrows  : {t1 - t0:7.3f} s")
print(f"apply     : {t2 - t1:7.3f} s")
print(f"vectorized: {t3 - t2:7.3f} s   ({(t1 - t0) / (t3 - t2):,.0f}x faster than iterrows)")
print("same result:", np.allclose(r1, r3) and np.allclose(r2, r3))

iterrows  :   1.397 s
apply     :   0.465 s
vectorized:   0.001 s   (1,931x faster than iterrows)
same result: True


In [37]:
def mem_mb(d):
    return d.memory_usage(deep=True).sum() / 1e6

before = mem_mb(tx)
slim = tx.assign(
    customer_id=tx["customer_id"].astype("int32"),
    amount=tx["amount"].astype("float32"),
    month=tx["month"].astype("int8"),
    hour=tx["hour"].astype("int8"),
    weekday=tx["weekday"].astype("category"),
)
print(f"memory {before:.2f} MB -> {mem_mb(slim):.2f} MB")

memory 0.71 MB -> 0.16 MB


## 15. Applications

In [38]:
# Application 1 : RFM customer segmentation (marketing) + expert rules
as_of = pd.Timestamp("2026-11-01")
rfm = tx.groupby("customer_id").agg(
    last_purchase=("ts", "max"),
    frequency=("ts", "size"),
    monetary=("amount", "sum"),
)
rfm["recency_days"] = (as_of - rfm["last_purchase"]).dt.days

rfm["R"] = pd.qcut(rfm["recency_days"], 4, labels=[4, 3, 2, 1]).astype(int)          # recent = high score
rfm["F"] = pd.qcut(rfm["frequency"].rank(method="first"), 4, labels=[1, 2, 3, 4]).astype(int)
rfm["M"] = pd.qcut(rfm["monetary"], 4, labels=[1, 2, 3, 4]).astype(int)

rfm["segment"] = np.select(
    [(rfm["R"] >= 3) & (rfm["F"] >= 3) & (rfm["M"] >= 3),
     (rfm["R"] <= 2) & (rfm["F"] >= 3),
     (rfm["R"] >= 3) & (rfm["F"] <= 2),
     (rfm["R"] == 1)],
    ["champions", "at_risk", "promising", "lost"],
    default="regular",
)
print(rfm[["recency_days", "frequency", "monetary", "R", "F", "M", "segment"]].head())
print(rfm.groupby("segment").agg(customers=("R", "size"), avg_recency=("recency_days", "mean"),
                                 avg_freq=("frequency", "mean"), avg_money=("monetary", "mean")).round(1))

             recency_days  frequency  monetary  R  F  M    segment
customer_id                                                       
1001                  207          1     171.0  1  1  1       lost
1002                   55          2     867.0  2  1  2    regular
1003                  104          2    1183.0  1  1  3       lost
1004                   41          2     723.0  3  1  2  promising
1005                   20          7    1785.0  4  4  4  champions
           customers  avg_recency  avg_freq  avg_money
segment                                               
at_risk          368         93.4       5.2     1578.2
champions        512         21.8       6.0     1854.8
lost             363        171.0       2.2      656.2
promising        379         23.2       2.8      817.4
regular          346         60.1       3.3      822.6


In [39]:
# Application 2 : demand forecasting features per store , without leakage
rng = np.random.default_rng(11)
days = pd.date_range("2026-01-01", periods=300, freq="D")
weekend_boost = np.where(days.dayofweek.isin([3, 4]), 40, 0)          # Thursday / Friday
frames = []
for store, level in [("S1", 200), ("S2", 120), ("S3", 300)]:
    trend = np.linspace(0, 30, len(days))
    frames.append(pd.DataFrame({"date": days, "store": store,
                                "sales": level + trend + weekend_boost + rng.normal(0, 10, len(days))}))
sales = pd.concat(frames, ignore_index=True).sort_values(["store", "date"], ignore_index=True)

wrong_lag = sales["sales"].shift(1)                                     # crosses store boundaries !
g = sales.groupby("store")["sales"]
sales["lag_1"] = g.shift(1)
sales["lag_7"] = g.shift(7)
sales["ma_7"] = g.transform(lambda s: s.shift(1).rolling(7).mean())
sales["weekend"] = sales["date"].dt.dayofweek.isin([3, 4]).astype(int)

first_s2 = sales.index[sales["store"] == "S2"][0]
print("first S2 row : wrong lag =", round(wrong_lag[first_s2], 1), "(last S1 value) | correct lag =", sales.loc[first_s2, "lag_1"])
print(sales.dropna().head(3).round({"sales": 1, "lag_1": 1, "lag_7": 1, "ma_7": 1}))

first S2 row : wrong lag = 226.3 (last S1 value) | correct lag = nan
        date store  sales  lag_1  lag_7   ma_7  weekend
7 2026-01-08    S1  240.1  206.3  240.3  214.4        1
8 2026-01-09    S1  248.3  240.1  253.7  214.3        1
9 2026-01-10    S1  182.4  248.3  212.4  213.6        0


In [40]:
data = sales.dropna().copy()
cutoff = pd.Timestamp("2026-09-01")
train, test = data[data["date"] < cutoff], data[data["date"] >= cutoff]       # TIME split

feats = ["lag_1", "lag_7", "ma_7", "weekend"]
def design(d):
    return np.column_stack([d[feats].to_numpy(), np.ones(len(d))])
w, *_ = np.linalg.lstsq(design(train), train["sales"].to_numpy(), rcond=None)

pred = design(test) @ w
rmse = lambda e: np.sqrt(np.mean(e ** 2))
print("train rows", len(train), "| test rows", len(test))
print("RMSE linear model  :", round(rmse(test["sales"] - pred), 2))
print("RMSE naive lag_1   :", round(rmse(test["sales"] - test["lag_1"]), 2))
print("RMSE seasonal lag_7:", round(rmse(test["sales"] - test["lag_7"]), 2))
print("weights:", {k: round(float(v), 2) for k, v in zip(feats + ["bias"], w)})

train rows 708 | test rows 171
RMSE linear model  : 10.55
RMSE naive lag_1   : 26.32
RMSE seasonal lag_7: 14.21
weights: {'lag_1': -0.02, 'lag_7': 0.0, 'ma_7': 1.01, 'weekend': 41.38, 'bias': -11.05}


In [41]:
# Application 3 : target-encoding LEAKAGE on a feature with NO real signal
rng = np.random.default_rng(3)
m = 6000
d = pd.DataFrame({"zip": rng.integers(0, 600, m), "y": (rng.random(m) < 0.3).astype(int)})   # y is pure noise
is_test = rng.random(m) < 0.3
tr, te = d[~is_test], d[is_test]

def auc(score, y):
    r = pd.Series(score).rank().to_numpy()
    pos = y == 1
    n_pos, n_neg = pos.sum(), (~pos).sum()
    return (r[pos].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)

leaky = d.groupby("zip")["y"].mean()                         # uses ALL rows , including test labels
safe = tr.groupby("zip")["y"].mean()                         # uses training rows only
prior = tr["y"].mean()

print("test AUC leaky encoding:", round(auc(te["zip"].map(leaky).to_numpy(), te["y"].to_numpy()), 3))
print("test AUC safe encoding :", round(auc(te["zip"].map(safe).fillna(prior).to_numpy(), te["y"].to_numpy()), 3))

test AUC leaky encoding: 0.688
test AUC safe encoding : 0.507


In [42]:
# Application 4 : sensor anomaly detection per machine (rolling z-score , past data only)
rng = np.random.default_rng(5)
frames = []
for mid, base in [("pump_A", 60.0), ("pump_B", 75.0), ("fan_C", 40.0)]:
    temp = base + np.cumsum(rng.normal(0, 0.05, 500)) + rng.normal(0, 0.5, 500)
    frames.append(pd.DataFrame({"machine": mid, "t": np.arange(500), "temp": temp}))
sensors = pd.concat(frames, ignore_index=True)

spikes = sensors.sample(6, random_state=1).index
sensors.loc[spikes, "temp"] += 6.0
sensors["injected"] = sensors.index.isin(spikes)

g = sensors.groupby("machine")["temp"]
mu = g.transform(lambda s: s.shift(1).rolling(50, min_periods=20).mean())
sd = g.transform(lambda s: s.shift(1).rolling(50, min_periods=20).std())
sensors["z"] = (sensors["temp"] - mu) / sd
sensors["alarm"] = sensors["z"].abs() > 4

print(sensors[sensors["alarm"]][["machine", "t", "temp", "z", "injected"]].round(2))
print(pd.crosstab(sensors["injected"], sensors["alarm"]))

     machine    t   temp      z  injected
75    pump_A   75  66.19  12.52      True
91    pump_A   91  65.58   5.64      True
330   pump_A  330  65.32  11.68      True
774   pump_B  274  82.56  13.01      True
1264   fan_C  264  45.92  10.26      True
1349   fan_C  349  45.03  11.40      True
alarm     False  True 
injected              
False      1494      0
True          0      6


In [43]:
# Application 5 : an expert system whose RULES are DATA (a DataFrame)
applicants = pd.DataFrame({
    "applicant": ["A1", "A2", "A3", "A4", "A5", "A6"],
    "income": [45, 12, 80, 30, 25, 60],                 # million toman / month
    "debt_ratio": [0.20, 0.55, 0.10, 0.45, 0.30, 0.65],
    "late_payments": [0, 3, 0, 1, 0, 0],
    "years_employed": [5, 1, 12, 0, 3, 8],
})
rules = pd.DataFrame({
    "priority": [1, 2, 3, 4, 5],
    "condition": ["late_payments >= 3",
                  "debt_ratio > 0.6",
                  "income >= 40 and debt_ratio < 0.3 and late_payments == 0",
                  "years_employed < 1",
                  "debt_ratio <= 0.35"],
    "decision": ["reject", "reject", "approve", "manual_review", "approve"],
    "reason": ["payment history", "too much debt", "strong profile", "employment too short", "acceptable debt"],
}).sort_values("priority")

fired = pd.DataFrame({r.priority: applicants.eval(r.condition) for r in rules.itertuples()})
fired[99] = True                                                   # default rule
first = fired.idxmax(axis=1)                                      # first True column = highest priority
lookup = rules.set_index("priority")
applicants["decision"] = first.map(lookup["decision"]).fillna("manual_review")
applicants["reason"] = first.map(lookup["reason"]).fillna("no rule matched")
print(fired.astype(int))
print(applicants)

   1   2   3   4   5   99
0   0   0   1   0   1   1
1   1   0   0   0   0   1
2   0   0   1   0   1   1
3   0   0   0   1   0   1
4   0   0   0   0   1   1
5   0   1   0   0   0   1
  applicant  income  debt_ratio  late_payments  years_employed       decision                reason
0        A1      45        0.20              0               5        approve        strong profile
1        A2      12        0.55              3               1         reject       payment history
2        A3      80        0.10              0              12        approve        strong profile
3        A4      30        0.45              1               0  manual_review  employment too short
4        A5      25        0.30              0               3        approve       acceptable debt
5        A6      60        0.65              0               8         reject         too much debt


In [44]:
# Application 6 : evaluate a churn model by SLICE (city , plan) , not only overall
c = customers.dropna(subset=["monthly_spend"]).copy()
risk = 0.45 * c["support_calls"] - 0.05 * c["tenure_months"] + 0.5 * (c["plan"] == "basic")
c["pred"] = risk > risk.quantile(0.6)

print(pd.crosstab(c["churned"], c["pred"], rownames=["actual"], colnames=["predicted"]))

def metrics(g):
    tp = (g["pred"] & g["churned"]).sum()
    return pd.Series({
        "n": len(g),
        "accuracy": (g["pred"] == g["churned"]).mean(),
        "precision": tp / g["pred"].sum(),
        "recall": tp / g["churned"].sum(),
    })

print("overall:", metrics(c).round(3).to_dict())
print(pd.DataFrame({k: metrics(g) for k, g in c.groupby("plan", observed=True)}).T.round(3))
print(pd.DataFrame({k: metrics(g) for k, g in c.groupby("city")}).T.round(3))

predicted  False  True 
actual                 
False       1035    479
True         118    288
overall: {'n': 1920.0, 'accuracy': 0.689, 'precision': 0.375, 'recall': 0.709}
              n  accuracy  precision  recall
basic    1148.0     0.642      0.414   0.802
pro       573.0     0.754      0.227   0.408
premium   199.0     0.774      0.286   0.444
             n  accuracy  precision  recall
Isfahan  273.0     0.700      0.439   0.681
Mashhad  283.0     0.693      0.398   0.703
Shiraz   315.0     0.689      0.316   0.643
Tabriz   289.0     0.651      0.308   0.673
Tehran   760.0     0.699      0.393   0.759


## Exercises
1. build two Series with different indexes and compute their difference with `sub(..., fill_value=0)`.
2. select customers from Tehran or Shiraz , on the `pro` plan , with more than 3 support calls , using both `.loc` and `.query`.
3. impute missing `age` with the median of each `city` : is it better than the global median ? (use `truth`)
4. clean `["  ۱۲,۵۰۰ ", "7.5k", "-", "3000"]` into numbers ("7.5k" -> 7500).
5. make a table of the churn rate for every (plan , tenure group) pair , with tenure groups `<12 , 12-35 , 36+` months.
6. merge `rfm` with `customers` : which merge type do you need ? how many customers have NO transactions ?
7. in Application 2 add a `lag_14` feature : does the test RMSE improve ?
8. in Application 3 , compute the TRAIN AUC of the leaky encoding : what does it tell you ?
9. in Application 5 add a rule `income < 15 -> reject` with priority 0 : which decisions change ?
10. in Application 6 , choose the threshold quantile so that overall recall is at least 0.7 . what happens to precision ?